In [2]:
"""
agropulse_xai/run_pipeline_harmonizer.py

Script del rol Matemático adaptado para Google Colab y Google Drive.
Ejecuta la armonización geoespacial (downscaling 30m) y la optimización TOPSIS.
"""

import os
import sys
import numpy as np
import pandas as pd

# 1. Conectar Google Drive y configurar sys.path para importar core/
try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR = '/content/drive/MyDrive/agropulse_xai'
    print("[INFO] Montado Google Drive en Colab correctamente.")
except ImportError:
    BASE_DIR = os.path.abspath(os.path.dirname(__file__))
    print("[INFO] Ejecutando en entorno local.")

CORE_DIR = os.path.join(BASE_DIR, 'core')
DATA_DIR = os.path.join(BASE_DIR, 'data')

if CORE_DIR not in sys.path:
    sys.path.append(CORE_DIR)

# Importar funciones del módulo harmonizer.py
from harmonizer import (
    downscale_smap_rootzone_30m,
    lapse_rate_correction_temperature,
    run_topsis_optimization
)

# 2. Rutas exactas de los 5 archivos en data/
PATH_D1_D3 = os.path.join(DATA_DIR, "D1_D3_grid30m_cdl_srtm.parquet")
PATH_D2    = os.path.join(DATA_DIR, "D2_ssurgo_soil.parquet")
PATH_D4    = os.path.join(DATA_DIR, "D4_hls_ndvi_bsi_series.parquet")
PATH_D5_D6 = os.path.join(DATA_DIR, "D5_D6_smap_gpm_daily.parquet")
PATH_D7    = os.path.join(DATA_DIR, "D7_nasa_power_daily.parquet")

PATH_EVAL       = os.path.join(DATA_DIR, "evaluated_metrics.parquet")
PATH_OUT_GRID   = os.path.join(DATA_DIR, "harmonized_grid_30m.parquet")
PATH_OUT_TOPSIS = os.path.join(DATA_DIR, "topsis_rankings.parquet")


def main():
    print("=== INICIANDO PIPELINE DE ARMONIZACIÓN Y DOWNSCALING (MATEMÁTICO) ===")

    # --------------------------------------------------------------------------
    # CARGA DE LOS 5 DATASETS REALES DESDE DATA/
    # --------------------------------------------------------------------------
    df_grid     = pd.read_parquet(PATH_D1_D3)
    df_soil     = pd.read_parquet(PATH_D2)
    df_hls      = pd.read_parquet(PATH_D4)
    df_smap_gpm = pd.read_parquet(PATH_D5_D6)
    df_power    = pd.read_parquet(PATH_D7)

    print(f"-> Cargados {len(df_grid)} píxeles a 30m desde D1_D3.")
    print(f"-> Cargadas {len(df_soil)} filas edáficas desde D2.")
    print(f"-> Cargadas {len(df_hls)} observaciones fenológicas desde D4.")
    print(f"-> Cargadas {len(df_smap_gpm)} series diarias desde D5_D6 y {len(df_power)} desde D7.")

    # --------------------------------------------------------------------------
    # PASO 1: MODULACIÓN EDÁFICA SSURGO (D2) A LA GRILLA 30M (D1_D3)
    # --------------------------------------------------------------------------
    awc_promedio = df_soil['agua_disponible_awc'].mean() if 'agua_disponible_awc' in df_soil.columns else 0.18
    df_spatial_30m = df_grid.copy()
    df_spatial_30m['ssurgo_awc'] = awc_promedio

    # --------------------------------------------------------------------------
    # PASO 2: DOWNSCALING DE HUMEDAD SMAP (D5_D6 -> 30M)
    # --------------------------------------------------------------------------
    smap_mean = df_smap_gpm['sm_rootzone'].dropna().mean()
    df_harmonized = downscale_smap_rootzone_30m(
        df_spatial_30m=df_spatial_30m,
        smap_val_9km=smap_mean,
        gamma_decay=0.5
    )

    # --------------------------------------------------------------------------
    # PASO 3: CORRECCIÓN TOPOGRÁFICA DE TEMPERATURA POR ELEVACIÓN (D7 + D1_D3)
    # --------------------------------------------------------------------------
    tmax_base = df_power['T2M_MAX'].mean()
    z_base = df_grid['elevation_m'].mean()

    df_harmonized['power_tmax_30m'] = lapse_rate_correction_temperature(
        t_interp=tmax_base,
        z_30m=df_grid['elevation_m'].values,
        z_coarse=z_base,
        gamma_lapse=0.0065
    )

    df_harmonized['gpm_precip_30m'] = df_smap_gpm['precipitation'].mean()

    # --------------------------------------------------------------------------
    # PASO 4: EXPORTACIÓN DE GRILLA ARMONIZADA (30M)
    # --------------------------------------------------------------------------
    cols_export = [
        'pixel_id', 'lon', 'lat', 'elevation_m', 'slope_pct',
        'ssurgo_awc', 'smap_sm_30m', 'gpm_precip_30m', 'power_tmax_30m'
    ]
    df_export_grid = df_harmonized[cols_export]
    df_export_grid.to_parquet(PATH_OUT_GRID, index=False)
    print(f"[OK] Generado exitosamente: {PATH_OUT_GRID} ({len(df_export_grid)} filas)")

    # --------------------------------------------------------------------------
    # PASO 5: OPTIMIZACIÓN MULTICRITERIO TOPSIS (SI EXISTEN MÉTRICAS EVALUADAS)
    # --------------------------------------------------------------------------
    if os.path.exists(PATH_EVAL):
        df_eval = pd.read_parquet(PATH_EVAL)
        pesos_defecto = {'water': 4.0, 'erosion': 4.0, 'nitrogen': 3.0, 'profit': 4.0, 'pest': 3.0}

        df_topsis = run_topsis_optimization(df_eval, pesos_defecto)
        df_topsis.to_parquet(PATH_OUT_TOPSIS, index=False)
        print(f"[OK] Generado exitosamente: {PATH_OUT_TOPSIS}")
    else:
        print("[INFO] 'evaluated_metrics.parquet' aún no existe en /data. TOPSIS se ejecutará al consolidar los cálculos agronómicos.")

    print("=== PIPELINE FINALIZADO CORRECTAMENTE ===")


if __name__ == "__main__":
    main()

MessageError: Error: credential propagation was unsuccessful

In [3]:
import sys
import os
from google.colab import drive

# 1. Montar Google Drive
drive.mount('/content/drive')

# 2. Especificar la carpeta donde está harmonizer.py
# (Si lo guardaste dentro de la carpeta 'core', usa .../agropulse_xai/core, de lo contrario ajusta la ruta)
ruta_carpeta = '/content/drive/MyDrive/agropulse_xai/Notebooks'

# 3. Agregar la carpeta a sys.path para que Python la reconozca como módulo
if ruta_carpeta not in sys.path:
    sys.path.append(ruta_carpeta)

# 4. Ahora sí importas las funciones sin errores
from harmonizer import (
    downscale_smap_rootzone_30m,
    lapse_rate_correction_temperature,
    run_topsis_optimization
)

print("¡Módulo harmonizer importado con éxito!")

MessageError: Error: credential propagation was unsuccessful